## TOOLS


Models can request to call tolls that perform taks such as fetching data form a datavase, searchin the we or running code. Tools are pairing of :
1. A schema , including the name fo the tool, a description , and arguments definitions(often a JSON schema)
2. A function ro coroutine to execute.

import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY]=os.getenv("GROQ_API_KEY")

model=init_chat_model("openai/gpt-oss-120b")
response=model.invoke("How to 24 LPA package?")
response.content


In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-120b")
model.invoke("how to get 24LPA pkg")
response = model.invoke("hwo is the weather in pune")
response.content

'I’m not able to pull live weather data. To get the current conditions in Pune, you can check a reliable weather service such as:\n\n- **Weather.com** (search “Pune, India”)\n- **AccuWeather** (search “Pune, India”)\n- **Google** – just type “Pune weather” into the search bar\n- A weather app on your phone (e.g., the built‑in iOS/Android weather app)\n\nThese sources will give you up‑to‑date temperature, precipitation, humidity, and forecast information.'

### writting tool code


In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str)-> str:
    """ getting a weather of the location"""
    return f" It's sunny in {location}"

binding tools with model

In [4]:
model_with_tools=model.bind_tools([get_weather])

#### view tool calls made of model

In [8]:
res=model_with_tools.invoke("what is the weathe in pune?")

for tool_call in res.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args:{tool_call['args']}")

Tool: get_weather
Args:{'location': 'Pune'}


Tools execution Loop


In [9]:
#step1 : model generates tool calls
messages=[{"role":"user", "content": "what's the weather in Pune?" }]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)


#step2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)
final_response=model_with_tools.invoke(messages)
print(final_response.text)



It’s sunny in Pune today.


In [10]:
messages

[{'role': 'user', 'content': "what's the weather in Pune?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Pune. Need to call get_weather function with location "Pune".', 'tool_calls': [{'id': 'fc_5816f882-d0ee-465b-8631-f708b06ee1d9', 'function': {'arguments': '{"location":"Pune"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 127, 'total_tokens': 175, 'completion_time': 0.101555306, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.005411958, 'prompt_tokens_details': None, 'queue_time': 0.387771218, 'total_time': 0.106967264}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a157de-bee1-7565-9852-19d59a1964ec-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': 'fc_5816f882-d0